In [1]:
import axelrod as axl
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt


In [ ]:
R = 3
S = 0
P = 1

T_values = [3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]

games = [axl.Game(r=R, s=S, t=T, p=P) for T in T_values]

p = [axl.Fuzzy()] + [axl.ZDExtortion()] + [axl.ZDGTFT2()] + [axl.ZDExtort2()] + [axl.ZDGen2()] + [axl.GTFT()] + [axl.WinStayLoseShift()] + [axl.RiskyQLearner()] + [axl.ArrogantQLearner()] + [axl.HesitantQLearner()]  + [axl.CautiousQLearner()] + [axl.TitForTat()] + [axl.Random()]
dffs = []

for g in games:
    dfs = []
    for i in range(0,30):
        t = axl.Tournament(p, game=g, turns=200, repetitions=1)
        results = t.play(build_results=True)
        
        df = pd.DataFrame(
                {
                    "Name": [x.name for x in p],
                    "#Wins": list(map(lambda x: sum(x), results.wins)),
                    "TotalScore": list(map(lambda x: sum(x), results.scores)),
                    "CooperationRating" : results.cooperating_rating,
                    "GoodPartnerRating" : results.good_partner_rating,
                    "EigenJesusRating" : results.eigenjesus_rating,
                    "EigenMosesRating" : results.eigenmoses_rating,
                })

        df = df.sort_values("TotalScore", ascending=False)
        df = df.reset_index(drop=True)

        dfs.append(df)
    dffs.append(dfs)

Playing matches:   2%|▏         | 2/91 [00:00<00:43,  2.02it/s]C:\Users\Ognjen\axl_mab\Axelrod\axelrod\strategies\zero_determinant.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  s_min = -min((T - l) / (l - S), (l - S) / (T - l))
Playing matches:   2%|▏         | 2/91 [00:01<01:05,  1.37it/s]C:\Users\Ognjen\axl_mab\Axelrod\axelrod\strategies\zero_determinant.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  s_min = -min((T - l) / (l - S), (l - S) / (T - l))
Playing matches:   2%|▏         | 2/91 [00:01<01:16,  1.17it/s]C:\Users\Ognjen\axl_mab\Axelrod\axelrod\strategies\zero_determinant.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  s_min = -min((T - l) / (l - S), (l - S) / (T - l))
Playing matches:   2%|▏         | 2/91 [00:01<01:02,  1.42it/s]C:\Users\Ognjen\axl_mab\Axelrod\axelrod\strategies\zero_determinant.py:93: RuntimeWarning: divide by zero encountered in scalar divide
  s_min = -min((T - l) / (l - S), (l - S) / (T - 

In [3]:
writer = pd.ExcelWriter('axelord_t_values_2.xlsx', engine='xlsxwriter')

j = 0
for dfs in dffs:
    i = 0
    for df in dfs:
        df.to_excel(writer, sheet_name=(str(T_values[j]) + " (" + str(i) + ")"), index=False)
        i += 1
    j += 1

writer.close()

In [ ]:
finalDFS = []
t = 3
for dfs in dffs:
    combined = pd.concat(dfs)
        
    agg = combined.groupby('Name').agg({
            '#Wins': 'sum',
            'TotalScore': 'sum',
            'CooperationRating': 'mean',
            'GoodPartnerRating': 'mean',
            'EigenJesusRating': 'mean',
            'EigenMosesRating': 'mean'
    }).reset_index()
        
        # Compute %Wins and AvgScore
    num_dfs = len(dfs)
    total_matches = (len(p)-1) * num_dfs
    agg['%Wins'] = agg['#Wins'] / total_matches
    agg['AvgScore'] = agg['TotalScore'] / total_matches
    agg['T'] = t
        
        # Rename columns
    agg = agg.rename(columns={
        'CooperationRating': 'AvgCoopRating',
        'GoodPartnerRating': 'AvgGP',
        'EigenJesusRating': 'AvgEJ',
        'EigenMosesRating': 'AvgEM'
    })
        
        # Keep only desired columns
    result = agg[['Name', '%Wins', 'AvgScore', 'AvgCoopRating', 'AvgGP', 'AvgEJ', 'AvgEM', 'T']]
    result = result.sort_values(by="AvgScore", ascending=False)

    finalDFS.append(result)
    t+=0.5

writer = pd.ExcelWriter('axelord_t_values_2_FinalDFS.xlsx', engine='xlsxwriter')


j = 0
for df in finalDFS:
    df.to_excel(writer, sheet_name=("T = " + str(T_values[j])), index=False)
    j += 1

justAvg = pd.concat(finalDFS)

justAvg = justAvg[['Name', 'AvgScore', 'T']]

justAvg.to_excel(writer, sheet_name="FINAL", index=False)



writer.close()

[                   Name     %Wins    AvgScore  AvgCoopRating     AvgGP  \
11             ZD-GEN-2  0.033333  515.800000       0.902167  0.966667   
12            ZD-GTFT-2  0.016667  514.400000       0.911667  0.983333   
3                  GTFT  0.000000  513.650000       0.885083  1.000000   
8   Win-Stay Lose-Shift  0.433333  510.433333       0.657417  0.566667   
7           Tit For Tat  0.000000  505.233333       0.783833  1.000000   
2                 Fuzzy  0.350000  502.550000       0.772750  0.650000   
6        Risky QLearner  0.366667  485.550000       0.927583  0.633333   
4     Hesitant QLearner  0.333333  476.400000       0.923917  0.666667   
1     Cautious QLearner  0.350000  470.616667       0.927333  0.650000   
0     Arrogant QLearner  0.416667  464.866667       0.925917  0.583333   
9           ZD-Extort-2  0.866667  425.700000       0.422833  0.133333   
5                Random  0.716667  424.266667       0.503250  0.283333   
10         ZD-Extortion  0.983333  39